# 📁 Notebook : Document Loaders

**LangChain 1.0.5+ | Mixed Level Class**

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
1. Load documents from **PDF files** using PyPDFLoader
2. Load structured data from **CSV files**
3. Load JSON data from **API responses** or files
4. Scrape and load content from **web pages** (HTML)
5. Load **text files** and **markdown files**
6. **Batch process** multiple files using DirectoryLoader
7. Understand Document object structure

---

## 📖 Table of Contents

1. [Why Document Loaders?](#why-loaders)
2. [Document Object Structure](#document-structure)
3. [Loading PDF Files](#pdf-loading)
4. [Loading CSV Files](#csv-loading)
5. [Loading JSON Files](#json-loading)
6. [Loading Web Pages (HTML)](#html-loading)
7. [Loading Text and Markdown Files](#text-loading)
8. [Batch Loading with DirectoryLoader](#batch-loading)
9. [Comparison Table](#comparison)
10. [Best Practices](#best-practices)
11. [Summary & Exercises](#summary)

---

<a id="why-loaders"></a>
## 1. Why Document Loaders? 🤔

### 🔰 BEGINNER

**Document Loaders** are tools that help you convert files (PDFs, CSVs, web pages, etc.) into **Document objects** that LangChain can work with.

Think of them as **translators**:
- **Input**: Files in various formats (PDF, CSV, JSON, HTML)
- **Output**: Standardized Document objects with text content and metadata

### Why is this important?

Every RAG application needs to:
1. 📥 **Load** data from various sources
2. 🔄 **Convert** it to a standard format
3. 📊 **Extract** metadata (source, page number, etc.)
4. 🎯 **Prepare** it for embedding and retrieval

Document Loaders handle all of this automatically!

### 🎓 INTERMEDIATE

All document loaders in LangChain implement the same interface:
- `.load()`: Load all documents at once (returns list[Document])
- `.lazy_load()`: Load documents one at a time (generator, memory efficient)

This consistency makes it easy to switch between different data sources.

In [1]:
# Setup: Import required libraries
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify setup
print("✅ Environment loaded")
print(f"Current directory: {os.getcwd()}")
print(f"Sample data directory exists: {Path('sample_data').exists()}")

✅ Environment loaded
Current directory: /Users/srinivasaraotadela/Documents/rag-langchain
Sample data directory exists: True


<a id="document-structure"></a>
## 2. Document Object Structure 📄

### 🔰 BEGINNER

Every Document has two main parts:
1. **page_content**: The actual text (string)
2. **metadata**: Information about the document (dictionary)

Think of it like a book:
- **page_content** = The story
- **metadata** = The cover information (title, author, page number, etc.)

In [6]:
from langchain_core.documents import Document

# Create a sample document
doc = Document(
    page_content="This is the actual content of the document. It contains the text we want to process.",
    metadata={
        "source": "example.pdf",
        "page": 1,
        "author": "John Doe",
        "date": "2025-01-15"
    }
)

# Inspect the document
print("📄 Document Structure:")
print(f"\nType: {type(doc)}")
print(f"\nContent (first 100 chars): {doc.page_content[:100]}...")
print(f"\nMetadata: {doc.metadata}")
print(f"\nSource: {doc.metadata['source']}")
print(f"Page Number: {doc.metadata['page']}")
print(f"Author: {doc.metadata['author']}")
print(f"Date: {doc.metadata['date']}")

📄 Document Structure:

Type: <class 'langchain_core.documents.base.Document'>

Content (first 100 chars): This is the actual content of the document. It contains the text we want to process....

Metadata: {'source': 'example.pdf', 'page': 1, 'author': 'John Doe', 'date': '2025-01-15'}

Source: example.pdf
Page Number: 1
Author: John Doe
Date: 2025-01-15


<a id="pdf-loading"></a>
## 3. Loading PDF Files 📕

### 🔰 BEGINNER

**PyPDFLoader** is used to load PDF files. It:
- Extracts text from each page
- Creates one Document per page
- Automatically adds source and page number to metadata

### Example 1: Loading a Single PDF

In [7]:
from langchain_community.document_loaders import PyPDFLoader

# Load the "Attention is All You Need" paper (if it exists)
pdf_path = "pdfs/rag.pdf"

if Path(pdf_path).exists():
    print(f"Loading PDF: {pdf_path}")
    print("⏳ This may take a moment...\n")
    
    # Create loader
    loader = PyPDFLoader(pdf_path)
    
    # Load all pages
    documents = loader.load()
    
    print(f"✅ Loaded {len(documents)} pages\n")
    
    # Inspect first page
    print("📄 First Page:")
    print(f"   Content (first 200 chars): {documents[0].page_content[:200]}...")
    print(f"\n   Metadata: {documents[0].metadata}")
    
    # Inspect last page
    print(f"\n📄 Last Page (page {len(documents)}):")
    print(f"   Content (first 200 chars): {documents[-1].page_content[:200]}...")
    
else:
    print(f"❌ PDF not found: {pdf_path}")
    print("   Make sure the file exists in the project root")

Loading PDF: pdfs/rag.pdf
⏳ This may take a moment...

✅ Loaded 19 pages

📄 First Page:
   Content (first 200 chars): Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, W...

   Metadata: {'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'pdfs/rag.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}

📄 Last Page (page 19):
   Content (first 200 chars): Table 7: Number of instances in the datasets used. *A hidden subset of this data is used for evaluation
Task Train Development Test
Natural Questions 79169 8758 3611
TriviaQA 78786 8838 11314
WebQue

### 🎓 INTERMEDIATE: Lazy Loading for Large PDFs

For very large PDFs, use `.lazy_load()` to process one page at a time:

In [8]:
# Lazy loading example
if Path(pdf_path).exists():
    loader = PyPDFLoader(pdf_path)
    
    print("🔄 Lazy loading pages (memory efficient):")
    
    # Process first 3 pages only
    for i, page in enumerate(loader.lazy_load()):
        if i >= 5:  # Only process first 3 pages for demo
            break
        
        print(f"\nPage {i+1}:")
        print(f"  Length: {len(page.page_content)} characters")
        print(f"  Preview: {page.page_content[:100]}...")
    
    print("\n💡 Tip: Use lazy_load() for PDFs > 100 pages to save memory")

🔄 Lazy loading pages (memory efficient):

Page 1:
  Length: 2899 characters
  Preview: Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Alek...

Page 2:
  Length: 4545 characters
  Preview: The	Divine
Comedy	(x) q
Query
Encoder
q(x)
MIPS pθ
Generator pθ
(Parametric)
Margin-
alize
This	14th...

Page 3:
  Length: 3655 characters
  Preview: byθ that generates a current token based on a context of the previousi− 1 tokensy1:i−1, the original...

Page 4:
  Length: 4205 characters
  Preview: minimize the negative marginal log-likelihood of each target,∑
j− logp(yj|xj) using stochastic
gradi...

Page 5:
  Length: 4556 characters
  Preview: MSMARCO as an open-domain abstractive QA task. MSMARCO has some questions that cannot be
answered in...

💡 Tip: Use lazy_load() for PDFs > 100 pages to save memory


### Example 2: Loading Multiple PDFs from a Directory

In [9]:
# Load all PDFs from the pdfs/ directory
pdf_directory = "pdfs"

if Path(pdf_directory).exists():
    print(f"📂 Loading PDFs from: {pdf_directory}/\n")
    
    all_documents = []
    
    # Find all PDF files
    pdf_files = list(Path(pdf_directory).glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files:")
    
    for pdf_file in pdf_files:
        print(f"  - {pdf_file.name}")
        
        # Load each PDF
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        all_documents.extend(docs)
        
        print(f"    ✅ Loaded {len(docs)} pages")
    
    print(f"\n📊 Total: {len(all_documents)} pages from {len(pdf_files)} PDFs")
    
    # Show unique sources
    sources = set(doc.metadata['source'] for doc in all_documents)
    print(f"\nSources:")
    for source in sources:
        print(f"  - {Path(source).name}")
        
else:
    print(f"❌ Directory not found: {pdf_directory}")

📂 Loading PDFs from: pdfs/

Found 4 PDF files:
  - rag.pdf
    ✅ Loaded 19 pages
  - Understanding_Climate_Change.pdf
    ✅ Loaded 33 pages
  - attention_paper.pdf
    ✅ Loaded 15 pages
  - ragsurvey.pdf
    ✅ Loaded 21 pages

📊 Total: 88 pages from 4 PDFs

Sources:
  - attention_paper.pdf
  - ragsurvey.pdf
  - rag.pdf
  - Understanding_Climate_Change.pdf
